# 11. Factorial state spaces

![Factorial state space](../images/11_factorial_state_spaces.svg)

**Learning goals:** enumerate Cartesian products, separate factor subsets from their assignments, generate partial queries without duplicates, preserve exact probabilities, and compute a partial-information ceiling. The notebook uses only synthetic binary factors.

In [ ]:
from fractions import Fraction
from itertools import combinations, product
import numpy as np
import matplotlib.pyplot as plt

SEED = 11
rng = np.random.default_rng(SEED)
np.set_printoptions(linewidth=90)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Cartesian products and shapes

For factor domains $\mathcal V_1,\ldots,\mathcal V_d$, the state space is $\Omega=\mathcal V_1\times\cdots\times\mathcal V_d$. Its size is the product of domain sizes. A binary state is a row vector in $\{0,1\}^d$. Stacking all states produces an array of shape `(number of states, d)`.

In [ ]:
d = 3
states = np.array(list(product([0, 1], repeat=d)), dtype=np.int8)
expected_count = 2 ** d
assert states.shape == (expected_count, d)
assert np.unique(states, axis=0).shape[0] == expected_count
print("states shape:", states.shape)
print(states)

## 2. Bit-vector indexing

The binary vector $(x_0,\ldots,x_{d-1})$ has integer index $\sum_j x_j2^{d-1-j}$. Matrix multiplication computes every index at once. This is clearer and faster than a Python loop for a materialized state table. For very many binary rows, `np.packbits` provides compact storage.

In [ ]:
bit_weights = 2 ** np.arange(d - 1, -1, -1)
state_indices = states @ bit_weights
assert np.array_equal(state_indices, np.arange(2 ** d))
packed = np.packbits(states, axis=1, bitorder="big")
print("weights:", bit_weights, "indices:", state_indices.tolist())
print("packed storage shape:", packed.shape)

## 3. Factor subsets, complements, and queries

A subset $S$ names known coordinates. Its complement $S^c$ contains the remaining coordinates relative to the full index set. A query is `(S, values)`. For binary factors, every coordinate has three query statuses: omitted, fixed to 0, or fixed to 1. Therefore there are $3^d$ partial queries, including the empty query.

In [ ]:
def all_subsets(num_factors):
    for size in range(num_factors + 1):
        yield from combinations(range(num_factors), size)

def binary_queries(num_factors):
    for subset in all_subsets(num_factors):
        for assignment in product([0, 1], repeat=len(subset)):
            yield subset, assignment

queries = list(binary_queries(d))
assert len(queries) == 3 ** d
assert len(set(queries)) == len(queries)
S = (0, 2)
Sc = tuple(j for j in range(d) if j not in S)
assert Sc == (1,)
print(f"{len(queries)} unique queries; S={S}, complement={Sc}")

## 4. Exact partial-information ceilings

A query that reveals factors 0 and 2 leaves factor 1 unknown. With a uniform prior, two complete states match each observed assignment. The Bayes-optimal success probability is therefore exactly $1/2$. `fractions.Fraction` keeps this combinatorial result exact. It is ideal for small counts, but slower than vectorized floating-point arrays.

In [ ]:
def compatible_rows(table, subset, assignment):
    subset = np.asarray(subset, dtype=int)
    assignment = np.asarray(assignment, dtype=table.dtype)
    return np.all(table[:, subset] == assignment, axis=1)

assignments = list(product([0, 1], repeat=len(S)))
group_sizes = [int(compatible_rows(states, S, a).sum()) for a in assignments]
ceiling = sum((Fraction(size, len(states)) * Fraction(1, size)
               for size in group_sizes), start=Fraction(0, 1))
assert group_sizes == [2, 2, 2, 2]
assert ceiling == Fraction(1, 2)
print("compatible states per query:", group_sizes)
print("exact average ceiling:", ceiling, "=", float(ceiling))

## 5. Visualize one query

The Boolean mask below is vectorized across every state. `np.all(..., axis=1)` reduces coordinate comparisons to one compatibility flag per row. We plot the state indices and highlight the two that match query `1 ? 1`.

In [ ]:
mask = compatible_rows(states, S, (1, 1))
colors = np.where(mask, "#d89b21", "#c9d8e6")
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.bar(state_indices, np.ones(len(states)), color=colors, edgecolor="#52606d")
ax.set(xticks=state_indices, xticklabels=["".join(map(str, row)) for row in states],
       yticks=[], xlabel="complete state", title="States compatible with query 1 ? 1")
ax.set_ylim(0, 1.25)
plt.tight_layout()
plt.show()
assert state_indices[mask].tolist() == [5, 7]

## Exercises and takeaways

1. Change `d` to 4. Predict the numbers of complete states and partial queries before running.
2. Reveal only factor 0. Compute the ceiling. Then reveal no factors.
3. Replace one binary domain with three values and verify that state counts multiply.

**Brief answers:** 4 binary factors give 16 states and 81 queries. Revealing one of three binary factors leaves 4 compatible states, so the ceiling is $1/4$; revealing none gives $1/8$. A ternary factor multiplies the corresponding state count by 3 rather than 2.

**Takeaway:** factorial spaces separate complete states from partial observations. Query ambiguity sets a ceiling that no classifier can exceed without additional information.

## Continue learning

[Previous notebook: 10](10_regularized_linear_estimation.ipynb) | [Lecture](../lectures/11_factorial_state_spaces.md) | [Curriculum](../README.md) | [Next notebook: 12](12_blockwise_distances_and_ranking.ipynb)